<h1><b>MASS DATA DEPORTATION

In [67]:
import pandas as pd
from fredapi import Fred

# 1. Initialize with your API Key
FRED_API_KEY = '94318e235b2c36c1a3159affe94b3ffc' # Get one at: https://fredaccount.stlouisfed.org/apikeys
fred = Fred(api_key= FRED_API_KEY)


# 2. Define the series we want to track
series_dict = {
    # --- TARGETS (The things you want to predict) ---
    'CPI': 'CPIAUCSL',        # All items
    'CPI_Core': 'CPILFESL',            # All items less Food and Energy
    'PCE_Price_Index': 'PCEPI',        # The Fed's favorite inflation metric

    # # --- MONETARY & INTEREST RATES ---
    'M2_Money_Supply': 'M2SL',         # Currency in circulation
    'Fed_Funds_Rate': 'FEDFUNDS',      # Base interest rate
    '10Y_Treasury_Yield': 'GS10',      # Long-term interest rates
    '30Y_Mortgage_Rate': 'MORTGAGE30US', # Housing market pressure

    # --- LABOR MARKET (Phillips Curve indicators) ---
    'Unemployment_Rate': 'UNRATE',     # Labor slack
    'Job_Openings_JOLTS': 'JTSJOL',    # Labor demand
    'Average_Hourly_Earnings': 'CES0500000003', # Wage-push inflation

    # # --- ENERGY & COMMODITIES (Cost-push drivers) ---
    'Crude_Oil_WTI': 'DCOILWTICO',       # Primary energy cost 
    'Gasoline_Prices': 'GASREGW',      # Direct consumer impact
    'Producer_Price_Index': 'PPIACO',  # Wholesale inflation (leads CPI)

    # --- EXPECTATIONS (Self-fulfilling prophecy) ---
    'Inflation_Expectations_5Y': 'T5YIFR', # Market-based expectations
    'University_of_Michigan_Expectations': 'MICH', # Consumer-based survey

    # --- SUPPLY CHAIN & HOUSING ---
    'Housing_Starts': 'HOUST',         # Supply of new homes
    'Retail_Sales': 'RSXFS',           # Consumer demand strength

    # --- GLOBAL/EXTERNAL ---
    'US_Dollar_Index': 'DTWEXAFEGS',        # Measures USD strength; a stronger dollar usually lowers import costs and cools inflation.
    'Import_Price_Index': 'IR',             # Tracks prices of goods bought by U.S. residents from abroad; direct signal of imported inflation.
    
    # --- HOUSING DEEP DIVE ---
    'Home_Price_Index': 'CSUSHPISA',        # Case-Shiller Index; tracks residential real estate prices (a massive lead indicator for 'Rent' inflation).
    'Lumber_Prices': 'WPU081',              # PPI for Lumber; early warning sign for construction costs and future housing price spikes.
    
    # --- DEBT & STRESS ---
    'Consumer_Credit': 'TOTALSL',           # Total outstanding consumer debt; high levels suggest sustained demand even as cash thins out.
    'Financial_Stress_Index': 'STLFSI4',    # Captures market strain; high stress often leads to a drop in the 'Velocity of Money' and cooling prices.
    
    # --- ALTERNATIVE INFLATION VIEWS ---
    'Sticky_CPI': 'STICKCPIM157SFRBATL',    # Prices that change slowly (e.g., insurance, education); reflects long-term inflation expectations.
    'Flexible_CPI': 'FLEXCPIM157SFRBATL',  # Prices that change quickly (e.g., fuel, clothes); reacts instantly to economic shocks.
    'Median_CPI': 'MEDCPIM157SFRBCLE',       # Strips out extreme price changes in any single month to find the "true" underlying trend.

    # --- FINANCIAL MARKETS ---
    'SP500': 'SP500',                       # S&P 500 Index; reflects investor confidence and future growth expectations.
    'VIX_Volatility_Index': 'VIXCLS',       # The "Fear Index"; high volatility often precedes shifts in monetary policy.
    'NASDAQ_Index': 'NASDAQ100',            # Tech-heavy index; highly sensitive to interest rate and inflation shifts.
    'Corporate_Bond_Spread': 'BAMLH0A0HYM2' # High Yield spread; signals credit risk and potential economic tightening.
}

In [69]:
def get_monthly_economic_data(series_dict, start_date='1967-03-01'):
    data_frames = []
    
    for name, series_id in series_dict.items():
        print(f"Fetching {name} ({series_id})...")
        try:
            series_data = fred.get_series(series_id, observation_start=start_date)
            df = pd.DataFrame(series_data, columns=[name])
            df.index = pd.to_datetime(df.index)
            df = df.resample('MS').mean()
            data_frames.append(df)
        except Exception as e:
            print(f"Fout bij ophalen {name}: {e}")

    # Samenvoegen
    final_df = pd.concat(data_frames, axis=1).sort_index()
    results_df = pd.DataFrame(index=final_df.index)

    # 1. Bereken MoM groeipercentage voor ALLE kolommen (Features)
    for col in final_df.columns:
        results_df[f"{col}_MoM_Pct"] = final_df[col].pct_change(fill_method=None) * 100

    # 2. HET LABEL: MoM Inflatie van de VOLGENDE maand (Shift -1)
    # We pakken de huidige MoM inflatie en schuiven die één stap terug in de tijd.
    # Hierdoor staat op de rij van 'januari' het inflatiecijfer van 'februari'.
    if 'CPI_MoM_Pct' in results_df.columns:
        results_df['LABEL_Next_Month_Inflation'] = results_df['CPI_MoM_Pct'].shift(-1)

    # 3. NA's invullen met gemiddelde
    results_df = results_df.fillna(results_df.mean())

    # We verwijderen de allerlaatste rij omdat we daarvoor geen 'volgende maand' label hebben
    # En we verwijderen de eerste rij (NaN door pct_change)
    return results_df.iloc[1:-1]

In [70]:
mass_data = get_monthly_economic_data(series_dict)


Fetching CPI (CPIAUCSL)...
Fetching CPI_Core (CPILFESL)...
Fetching PCE_Price_Index (PCEPI)...
Fetching M2_Money_Supply (M2SL)...
Fetching Fed_Funds_Rate (FEDFUNDS)...
Fetching 10Y_Treasury_Yield (GS10)...
Fetching 30Y_Mortgage_Rate (MORTGAGE30US)...
Fetching Unemployment_Rate (UNRATE)...
Fetching Job_Openings_JOLTS (JTSJOL)...
Fetching Average_Hourly_Earnings (CES0500000003)...
Fetching Crude_Oil_WTI (DCOILWTICO)...
Fetching Gasoline_Prices (GASREGW)...
Fetching Producer_Price_Index (PPIACO)...
Fetching Inflation_Expectations_5Y (T5YIFR)...
Fetching University_of_Michigan_Expectations (MICH)...
Fetching Housing_Starts (HOUST)...
Fetching Retail_Sales (RSXFS)...
Fetching US_Dollar_Index (DTWEXAFEGS)...
Fetching Import_Price_Index (IR)...
Fetching Home_Price_Index (CSUSHPISA)...
Fetching Lumber_Prices (WPU081)...
Fetching Consumer_Credit (TOTALSL)...
Fetching Financial_Stress_Index (STLFSI4)...
Fetching Sticky_CPI (STICKCPIM157SFRBATL)...
Fetching Flexible_CPI (FLEXCPIM157SFRBATL)...
Fe

In [71]:
display(mass_data.tail())


,CPI_MoM_Pct,CPI_Core_MoM_Pct,PCE_Price_Index_MoM_Pct,M2_Money_Supply_MoM_Pct,Fed_Funds_Rate_MoM_Pct,10Y_Treasury_Yield_MoM_Pct,30Y_Mortgage_Rate_MoM_Pct,Unemployment_Rate_MoM_Pct,Job_Openings_JOLTS_MoM_Pct,Average_Hourly_Earnings_MoM_Pct,...,Consumer_Credit_MoM_Pct,Financial_Stress_Index_MoM_Pct,Sticky_CPI_MoM_Pct,Flexible_CPI_MoM_Pct,Median_CPI_MoM_Pct,SP500_MoM_Pct,VIX_Volatility_Index_MoM_Pct,NASDAQ_Index_MoM_Pct,Corporate_Bond_Spread_MoM_Pct,LABEL_Next_Month_Inflation
2025-11-01,0.326957,0.322995,0.219753,0.207187,-5.134474,0.738916,-0.263831,0.230602,-4.518828,0.407056,...,0.064030,-27.485656,0.000000,-18.350421,0.000000,0.077112,9.305152,-0.071855,5.339534,0.297788
2025-12-01,0.297788,0.232900,0.330857,0.405445,-4.123711,1.222494,-0.761523,-2.222222,-4.323693,0.054054,...,0.311884,25.930027,144.688451,127.088557,121.072444,1.663575,-21.352681,1.374515,-6.138823,0.170843
2026-01-01,0.170843,0.295045,0.302545,0.367179,-2.150538,1.690821,-1.413570,-2.272727,10.534351,0.351162,...,0.150304,53.233691,26.421060,-218.915032,-24.663052,1.110408,4.057489,0.614711,-5.234203,0.267003
2026-02-01,0.267003,0.216050,0.375296,0.882100,0.000000,-1.900238,-0.901270,2.325581,-4.944751,0.376851,...,0.185697,-15.568867,-42.402317,-219.829152,-12.360536,-0.509670,18.715270,-2.298763,6.534291,0.865144
2026-03-01,0.865144,0.195795,0.286029,0.545070,0.000000,2.905569,2.149649,-2.272727,0.256536,0.241352,...,0.555822,-42.055905,-2.250797,471.681999,12.007571,-3.472497,33.268174,-2.523564,9.170979,0.326957


In [ ]:
#mass_data.dropna(inplace=True)
print("\nData Extraction Complete!")
# Save to Excel (.xlsx) 
# We keep index=True because the Index contains the Dates!
mass_data.to_excel('BigChungus.xlsx', index=True)

print("\nSaved to BigChungus.xlsx")


Data Extraction Complete!

Saved to mass_deportation_dataset.xlsx
